HW1 — CUDA Matrix Multiplication + Profiling

SID4=6359  SEED=6359  SLICE=359  HP_ID=5 (Schedule-long; not used by this task, reported per Sec 0.1)
CLS_A=9  CLS_B=5 (not used by this task, reported per Sec 0.1)

In [ ]:
!nvidia-smi

Sun Aug 30 19:08:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Write the CUDA source (`matmul.cu`)

Naive matmul kernel: each thread computes one output element `C[row][col]`; each 16x16 thread block computes a 16x16 tile of `C`. See in-file comments for the full block/thread explanation.

In [ ]:
%%writefile matmul.cu
// matmul.cu
// HW1 Part 3 — Matrix multiplication in CUDA C, benchmarked against a CPU baseline.
// SID4=6359 SEED=6359 SLICE=359 HP_ID=5 (Schedule-long; unused by this CUDA task, reported per Sec 0.1)
// CLS_A=9 CLS_B=5 (unused by this CUDA task, reported per Sec 0.1)
//
// Build (Colab GPU runtime):
//   nvcc -O3 -Xcompiler -fopenmp matmul.cu -o matmul
// Run:
//   ./matmul 256
//   ./matmul 1024
//   ./matmul 4096
//
// Blocks/threads design (see kernel comment below): each CUDA thread computes exactly
// one output element C[row][col]. Threads are grouped into BLOCK_SIZE x BLOCK_SIZE
// (16x16 = 256 threads) thread blocks; a block therefore computes a 16x16 tile of C.
// The grid is sized as ceil(N/16) x ceil(N/16) blocks so the whole N x N output is
// covered, including when N is not a multiple of 16 (bounds-checked in the kernel).


#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <cuda_runtime.h>

#define BLOCK_SIZE 16

// ---------------------------------------------------------------------------
// CUDA kernel: naive tiled-by-thread-block matrix multiplication, C = A * B
// A: N x N, B: N x N, C: N x N (square matrices, row-major, float32)
//
// Grid/block mapping:
//   - blockDim = (BLOCK_SIZE, BLOCK_SIZE) -> 256 threads per block
//   - gridDim  = (ceil(N/BLOCK_SIZE), ceil(N/BLOCK_SIZE)) blocks
//   - each block owns a BLOCK_SIZE x BLOCK_SIZE tile of the output matrix C
//   - within a block, thread (tx, ty) computes exactly one element:
//         col = blockIdx.x * BLOCK_SIZE + threadIdx.x
//         row = blockIdx.y * BLOCK_SIZE + threadIdx.y
//     that thread walks the full K-dimension (K = N here) doing a dot product
//     of A's row `row` against B's column `col`, then writes C[row][col].
// ---------------------------------------------------------------------------
__global__ void matmulKernel(const float* A, const float* B, float* C, int N) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < N && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < N; ++k) {
            acc += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = acc;
    }
}

static void cpuMatmul(const float* A, const float* B, float* C, int N) {
    #pragma omp parallel for
    for (int row = 0; row < N; ++row) {
        for (int col = 0; col < N; ++col) {
            float acc = 0.0f;
            for (int k = 0; k < N; ++k) {
                acc += A[row * N + k] * B[k * N + col];
            }
            C[row * N + col] = acc;
        }
    }
}

static void fillRandom(float* M, size_t count, unsigned seed) {
    srand(seed);
    for (size_t i = 0; i < count; ++i) {
        M[i] = static_cast<float>(rand()) / RAND_MAX;
    }
}

static double maxAbsDiff(const float* a, const float* b, size_t count) {
    double m = 0.0;
    for (size_t i = 0; i < count; ++i) {
        double d = fabs((double)a[i] - (double)b[i]);
        if (d > m) m = d;
    }
    return m;
}

#define CUDA_CHECK(call)                                                          \
    do {                                                                          \
        cudaError_t err = (call);                                                 \
        if (err != cudaSuccess) {                                                 \
            fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err),   \
                    __FILE__, __LINE__);                                          \
            exit(1);                                                              \
        }                                                                         \
    } while (0)

int main(int argc, char** argv) {
    const unsigned SEED = 6359; // SEED = SID4, used to seed matrix generation
    int N = 256;
    if (argc > 1) N = atoi(argv[1]);
    size_t bytes = (size_t)N * N * sizeof(float);
    printf("N = %d (%.2f MB per matrix)\n", N, bytes / (1024.0 * 1024.0));

    float *hA = (float*)malloc(bytes);
    float *hB = (float*)malloc(bytes);
    float *hC_cpu = (float*)malloc(bytes);
    float *hC_gpu = (float*)malloc(bytes);
    fillRandom(hA, (size_t)N * N, SEED);
    fillRandom(hB, (size_t)N * N, SEED + 1);

    // ---- CPU baseline timing ----
    auto cpuStart = std::chrono::high_resolution_clock::now();
    cpuMatmul(hA, hB, hC_cpu, N);
    auto cpuEnd = std::chrono::high_resolution_clock::now();
    double cpuMs = std::chrono::duration<double, std::milli>(cpuEnd - cpuStart).count();

    // ---- GPU: allocate device memory ----
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, bytes));
    CUDA_CHECK(cudaMalloc(&dB, bytes));
    CUDA_CHECK(cudaMalloc(&dC, bytes));

    cudaEvent_t h2dStart, h2dStop, kStart, kStop, d2hStart, d2hStop;
    cudaEventCreate(&h2dStart); cudaEventCreate(&h2dStop);
    cudaEventCreate(&kStart);   cudaEventCreate(&kStop);
    cudaEventCreate(&d2hStart); cudaEventCreate(&d2hStop);

    // Host -> Device transfer
    cudaEventRecord(h2dStart);
    CUDA_CHECK(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));
    cudaEventRecord(h2dStop);
    cudaEventSynchronize(h2dStop);
    float h2dMs = 0;
    cudaEventElapsedTime(&h2dMs, h2dStart, h2dStop);

    // Kernel launch: block/grid dimensions
    dim3 threadsPerBlock(BLOCK_SIZE, BLOCK_SIZE);
    dim3 numBlocks((N + BLOCK_SIZE - 1) / BLOCK_SIZE, (N + BLOCK_SIZE - 1) / BLOCK_SIZE);
    printf("threadsPerBlock = (%d, %d) = %d threads/block\n",
           threadsPerBlock.x, threadsPerBlock.y, threadsPerBlock.x * threadsPerBlock.y);
    printf("numBlocks       = (%d, %d) = %d blocks\n",
           numBlocks.x, numBlocks.y, numBlocks.x * numBlocks.y);

    // warm-up launch (not timed) to exclude first-launch overhead from the measurement
    matmulKernel<<<numBlocks, threadsPerBlock>>>(dA, dB, dC, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    cudaEventRecord(kStart);
    matmulKernel<<<numBlocks, threadsPerBlock>>>(dA, dB, dC, N);
    cudaEventRecord(kStop);
    cudaEventSynchronize(kStop);
    CUDA_CHECK(cudaGetLastError());
    float kernelMs = 0;
    cudaEventElapsedTime(&kernelMs, kStart, kStop);

    // Device -> Host transfer
    cudaEventRecord(d2hStart);
    CUDA_CHECK(cudaMemcpy(hC_gpu, dC, bytes, cudaMemcpyDeviceToHost));
    cudaEventRecord(d2hStop);
    cudaEventSynchronize(d2hStop);
    float d2hMs = 0;
    cudaEventElapsedTime(&d2hMs, d2hStart, d2hStop);

    double transferMs = h2dMs + d2hMs;
    double gpuEndToEndMs = kernelMs + transferMs;
    double speedup = cpuMs / gpuEndToEndMs;

    double diff = maxAbsDiff(hC_cpu, hC_gpu, (size_t)N * N);

    printf("\n---- RESULTS (N=%d) ----\n", N);
    printf("CPU time (ms):           %.4f\n", cpuMs);
    printf("GPU kernel time (ms):    %.4f\n", kernelMs);
    printf("H2D+D2H transfer (ms):   %.4f\n", transferMs);
    printf("GPU end-to-end (ms):     %.4f\n", gpuEndToEndMs);
    printf("Speedup (CPU/GPU e2e):   %.4fx\n", speedup);
    printf("Max abs diff CPU vs GPU: %e\n", diff);
    printf("CSV,%d,%.4f,%.4f,%.4f,%.4f\n", N, cpuMs, kernelMs, transferMs, speedup);

    cudaEventDestroy(h2dStart); cudaEventDestroy(h2dStop);
    cudaEventDestroy(kStart);   cudaEventDestroy(kStop);
    cudaEventDestroy(d2hStart); cudaEventDestroy(d2hStop);
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    free(hA); free(hB); free(hC_cpu); free(hC_gpu);
    return 0;
}


Writing matmul.cu


## 2. Compile

`-Xcompiler -fopenmp` parallelizes the CPU baseline across host cores so the N=4096 CPU run finishes in a reasonable time; it does not touch the GPU kernel.

In [ ]:
!nvcc -O3 -Xcompiler -fopenmp matmul.cu -o matmul
!ls -la matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
-rwxr-xr-x 1 root root 1007048 Aug 30 19:08 matmul


## 3. Time the kernel vs. CPU baseline for N = 256, 1024, 4096

Each run prints CPU time, GPU kernel time (isolated with `cudaEvent`s, excluding a warm-up launch), H2D+D2H transfer time, end-to-end GPU time, and speedup. A `CSV,...` line is also printed for easy parsing into `METRICS.md`.

In [ ]:
results = {}
for N in [256, 1024, 4096]:
    print(f"\n===== N = {N} =====")
    out = !./matmul {N}
    for line in out:
        print(line)
    csv_line = [l for l in out if l.startswith("CSV,")][0]
    _, n, cpu_ms, kernel_ms, transfer_ms, speedup = csv_line.split(",")
    results[int(n)] = {
        "cpu_ms": float(cpu_ms), "kernel_ms": float(kernel_ms),
        "transfer_ms": float(transfer_ms), "speedup": float(speedup),
    }
results


===== N = 256 =====
N = 256 (0.25 MB per matrix)
threadsPerBlock = (16, 16) = 256 threads/block
numBlocks       = (16, 16) = 256 blocks

---- RESULTS (N=256) ----
CPU time (ms):           27.4259
GPU kernel time (ms):    0.1509
H2D+D2H transfer (ms):   1.0478
GPU end-to-end (ms):     1.1988
Speedup (CPU/GPU e2e):   22.8787x
Max abs diff CPU vs GPU: 1.525879e-05
CSV,256,27.4259,0.1509,1.0478,22.8787

===== N = 1024 =====
N = 1024 (4.00 MB per matrix)
threadsPerBlock = (16, 16) = 256 threads/block
numBlocks       = (64, 64) = 4096 blocks

---- RESULTS (N=1024) ----
CPU time (ms):           4910.4850
GPU kernel time (ms):    9.1950
H2D+D2H transfer (ms):   5.5728
GPU end-to-end (ms):     14.7678
Speedup (CPU/GPU e2e):   332.5121x
Max abs diff CPU vs GPU: 9.155273e-05
CSV,1024,4910.4850,9.1950,5.5728,332.5121

===== N = 4096 =====
N = 4096 (64.00 MB per matrix)
threadsPerBlock = (16, 16) = 256 threads/block
numBlocks       = (256, 256) = 65536 blocks

---- RESULTS (N=4096) ----
CPU time (

{256: {'cpu_ms': 27.4259,
  'kernel_ms': 0.1509,
  'transfer_ms': 1.0478,
  'speedup': 22.8787},
 1024: {'cpu_ms': 4910.485,
  'kernel_ms': 9.195,
  'transfer_ms': 5.5728,
  'speedup': 332.5121},
 4096: {'cpu_ms': 693849.9563,
  'kernel_ms': 326.84,
  'transfer_ms': 82.153,
  'speedup': 1696.4839}}

In [ ]:
import pandas as pd
rows = []
for N, r in sorted(results.items()):
    rows.append({"Matrix size": N, "CPU (ms)": r["cpu_ms"], "GPU kernel (ms)": r["kernel_ms"],
                 "H2D+D2H (ms)": r["transfer_ms"], "Speedup": r["speedup"]})
timing_df = pd.DataFrame(rows)
timing_df.to_csv("cuda_timing.csv", index=False)
timing_df

,Matrix size,CPU (ms),GPU kernel (ms),H2D+D2H (ms),Speedup
0,256,27.4259,0.1509,1.0478,22.8787
1,1024,4910.4850,9.1950,5.5728,332.5121
2,4096,693849.9563,326.8400,82.1530,1696.4839


4. Profile with Nsight Systems / Nsight Compute / nvprof
Colab GPU runtime did not have nsys (Nsight Systems) or nvprof available (nsys: command not found).
Nsight Compute (ncu) was available and was used to produce the profiler output below.

Profiler used: Nsight Compute (ncu), since nsys was not available in this environment.

Note: the ncu run below re-executes ./matmul under profiling instrumentation, which adds
significant overhead to the reported kernel time (visible below as ~1.76s vs. the ~9.2ms
unprofiled kernel time for N=1024 reported in Section 3). The timing table and speedup
numbers in Section 3/METRICS.md use the unprofiled runs; the numbers in this section are
for profiler-derived kernel/memory-throughput analysis only, not for the reported timings.

In [ ]:
# Nsight Systems: produces a timeline that separates H2D/D2H memcpy from kernel execution.
!which nsys && nsys --version
!nsys profile --stats=true -o nsys_report_1024 ./matmul 1024

/bin/bash: line 1: nsys: command not found


In [ ]:
# If nvprof is available in your environment instead (older CUDA toolkits), use:
# !nvprof --print-gpu-trace ./matmul 1024
# Nsight Compute (kernel-level detail: occupancy, memory throughput, warp efficiency):
!which ncu && ncu --set basic -o ncu_report_1024 ./matmul 1024

/usr/local/cuda/bin/ncu
N = 1024 (4.00 MB per matrix)
==PROF== Connected to process 6254 (/content/matmul)
threadsPerBlock = (16, 16) = 256 threads/block
numBlocks       = (64, 64) = 4096 blocks
==PROF== Profiling "matmulKernel" - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "matmulKernel" - 1: 0%....50%....100% - 9 passes

---- RESULTS (N=1024) ----
CPU time (ms):           5626.2680
GPU kernel time (ms):    1758.7977
H2D+D2H transfer (ms):   5.8060
GPU end-to-end (ms):     1764.6037
Speedup (CPU/GPU e2e):   3.1884x
Max abs diff CPU vs GPU: 9.155273e-05
CSV,1024,5626.2680,1758.7977,5.8060,3.1884
==PROF== Disconnected from process 6254
==PROF== Report: /content/ncu_report_1024.ncu-rep


## 5. Crossover analysis

In this benchmark, GPU end-to-end time beats the CPU baseline at every tested size,
including the smallest (N=256): 1.20ms vs. 27.43ms CPU. The margin widens sharply with
size — by N=4096, GPU end-to-end is 409ms against a CPU baseline of over 11 minutes
(693,850ms). The crossover isn't at size zero because a fixed cost exists on the GPU
path regardless of matrix size: CUDA kernel launch overhead and the PCIe transfer of
A/B host-to-device and C device-to-host, which together cost roughly 1.05ms even at
N=256, where actual compute (0.15ms) is negligible. As N grows, compute scales as
O(N³) while transfer scales as O(N²), so compute quickly dominates the fixed overhead
and the GPU's massive parallelism (256–65,536 thread blocks here) pulls far ahead of
the CPU, whose cost also scales O(N³) but without that parallel offset.